In [2]:
# RAPORT JAKOŚCI - WCZYTANIE I PODGLĄD DANYCH

import pandas as pd
from google.colab import files

df = pd.read_csv("fuel_prices_clean.csv", sep=";")

display(df.head())

print("Wymiary danych:", df.shape)

,productName,effectiveDate,publishFrom,value,unit,productId,rok,miesiac,cena_za_litr
0,Pb95,2026-11-11,2026-11-11,5307.0,PLN/m3,41,2026,11,5.307
1,Pb95,2026-11-10,2026-11-10,5346.0,PLN/m3,41,2026,11,5.346
2,Pb95,2026-11-07,2026-11-07,5296.0,PLN/m3,41,2026,11,5.296
3,Pb95,2026-11-06,2026-11-06,5368.0,PLN/m3,41,2026,11,5.368
4,Pb95,2026-11-05,2026-11-05,5438.0,PLN/m3,41,2026,11,5.438


Wymiary danych: (5829, 9)


In [3]:
# RAPORT JAKOŚCI DANYCH - BRAKI I DUPLIKATY

print("=== PODSTAWOWE INFORMACJE ===")
print("Liczba wierszy:", len(df))
print("Liczba kolumn:", len(df.columns))

print("\n=== TYPY DANYCH ===")
print(df.dtypes)

print("\n=== BRAKUJĄCE WARTOŚCI ===")
braki = pd.DataFrame({
    "Liczba braków": df.isna().sum(),
    "Procent braków (%)": (df.isna().mean() * 100).round(2)
})
display(braki)

print("\n=== DUPLIKATY ===")
print("Identyczne wiersze:", df.duplicated().sum())

print("\n=== DUPLIKATY PALIWO + DATA ===")
print(
    df.duplicated(
        subset=["productName", "effectiveDate"]
    ).sum()
)

=== PODSTAWOWE INFORMACJE ===
Liczba wierszy: 5829
Liczba kolumn: 9

=== TYPY DANYCH ===
productName       object
effectiveDate     object
publishFrom       object
value            float64
unit              object
productId          int64
rok                int64
miesiac            int64
cena_za_litr     float64
dtype: object

=== BRAKUJĄCE WARTOŚCI ===


,Liczba braków,Procent braków (%)
productName,0,0.0
effectiveDate,0,0.0
publishFrom,2448,42.0
value,0,0.0
unit,0,0.0
productId,0,0.0
rok,0,0.0
miesiac,0,0.0
cena_za_litr,0,0.0



=== DUPLIKATY ===
Identyczne wiersze: 0

=== DUPLIKATY PALIWO + DATA ===
0


In [4]:
# RAPORT JAKOŚCI - POPRAWNOŚĆ DAT I CEN

daty = pd.to_datetime(df["effectiveDate"], errors="coerce")
ceny = pd.to_numeric(df["value"], errors="coerce")

print("=== KONTROLA DAT ===")
print("Nieprawidłowe daty:", daty.isna().sum())
print("Najstarsza data:", daty.min())
print("Najnowsza data:", daty.max())

print("\n=== KONTROLA CEN ===")
print("Nieprawidłowe ceny:", ceny.isna().sum())
print("Ceny zerowe:", (ceny == 0).sum())
print("Ceny ujemne:", (ceny < 0).sum())
print("Najniższa cena:", ceny.min())
print("Najwyższa cena:", ceny.max())

print("\n=== RODZAJE PALIW ===")
display(df["productName"].value_counts())

print("\n=== STATYSTYKI CEN WEDŁUG PALIWA ===")
display(
    df.groupby("productName")["value"]
      .agg(["count", "min", "median", "max"])
      .round(2)
)

=== KONTROLA DAT ===
Nieprawidłowe daty: 0
Najstarsza data: 2019-01-01 00:00:00
Najnowsza data: 2026-11-11 00:00:00

=== KONTROLA CEN ===
Nieprawidłowe ceny: 0
Ceny zerowe: 0
Ceny ujemne: 0
Najniższa cena: 2705.0
Najwyższa cena: 8311.0

=== RODZAJE PALIW ===


,count
productName,
Pb95,1943
Pb98,1943
ONEkodiesel,1943



=== STATYSTYKI CEN WEDŁUG PALIWA ===


,count,min,median,max
productName,,,,
ONEkodiesel,1943,2876.0,4769.0,7704.0
Pb95,1943,2705.0,4607.0,7519.0
Pb98,1943,2958.0,4994.0,8311.0


In [5]:
# RAPORT JAKOŚCI - WARTOŚCI ODSTAJĄCE (IQR)

wyniki = []

for paliwo, grupa in df.groupby("productName"):
    q1 = grupa["value"].quantile(0.25)
    q3 = grupa["value"].quantile(0.75)
    iqr = q3 - q1

    dolna = q1 - 1.5 * iqr
    gorna = q3 + 1.5 * iqr

    odstajace = grupa[
        (grupa["value"] < dolna) |
        (grupa["value"] > gorna)
    ]

    wyniki.append({
        "Paliwo": paliwo,
        "Liczba wartości odstających": len(odstajace),
        "Procent (%)": round(
            len(odstajace) / len(grupa) * 100, 2
        ),
        "Dolna granica IQR": round(dolna, 2),
        "Górna granica IQR": round(gorna, 2)
    })

raport_iqr = pd.DataFrame(wyniki)
display(raport_iqr)

,Paliwo,Liczba wartości odstających,Procent (%),Dolna granica IQR,Górna granica IQR
0,ONEkodiesel,82,4.22,2236.25,7102.25
1,Pb95,25,1.29,2230.00,7022.00
2,Pb98,20,1.03,2079.50,7791.50


In [6]:
# RAPORT JAKOŚCI - LUKI CZASOWE

df_czas = df.copy()

df_czas["effectiveDate"] = pd.to_datetime(
    df_czas["effectiveDate"], errors="coerce"
)

wyniki_luk = []

for paliwo, grupa in df_czas.groupby("productName"):
    grupa = grupa.sort_values("effectiveDate")

    roznice = grupa["effectiveDate"].diff().dt.days

    wyniki_luk.append({
        "Paliwo": paliwo,
        "Liczba obserwacji": len(grupa),
        "Liczba przerw > 1 dzień": int((roznice > 1).sum()),
        "Najdłuższa przerwa (dni)": roznice.max(),
        "Mediana odstępu (dni)": roznice.median()
    })

raport_luki = pd.DataFrame(wyniki_luk)
display(raport_luki)

,Paliwo,Liczba obserwacji,Liczba przerw > 1 dzień,Najdłuższa przerwa (dni),Mediana odstępu (dni)
0,ONEkodiesel,1943,437,6.0,1.0
1,Pb95,1943,437,6.0,1.0
2,Pb98,1943,437,6.0,1.0


In [7]:
# RAPORT JAKOŚCI - NAJDŁUŻSZE PRZERWY

pb95 = df_czas[
    df_czas["productName"] == "Pb95"
].sort_values("effectiveDate").copy()

pb95["Poprzednia data"] = pb95["effectiveDate"].shift(1)

pb95["Przerwa (dni)"] = (
    pb95["effectiveDate"] - pb95["Poprzednia data"]
).dt.days

najdluzsze = pb95[
    pb95["Przerwa (dni)"] >= 5
][["Poprzednia data", "effectiveDate", "Przerwa (dni)"]]

display(najdluzsze)

,Poprzednia data,effectiveDate,Przerwa (dni)
1864,2019-04-19,2019-04-24,5.0
1857,2019-05-03,2019-05-08,5.0
1625,2020-04-10,2020-04-15,5.0
1449,2020-12-24,2020-12-30,6.0
1383,2021-04-02,2021-04-07,5.0
1343,2021-06-03,2021-06-08,5.0
1201,2021-12-24,2021-12-30,6.0
1126,2022-04-15,2022-04-20,5.0
1117,2022-04-30,2022-05-05,5.0
1096,2022-06-02,2022-06-07,5.0


In [8]:
# RAPORT JAKOŚCI - SPRAWDZENIE JEDNOSTEK

print(df["unit"].value_counts(dropna=False))

unit
PLN/m3    5829
Name: count, dtype: int64


In [9]:
# RAPORT JAKOŚCI - DATY Z PRZYSZŁOŚCI

print("Daty z przyszłości:")
print((daty > pd.Timestamp.today()).sum())

Daty z przyszłości:
0


In [10]:
# RAPORT JAKOŚCI - SPRAWDZENIE CENY ZA LITR

roznica = (df["cena_za_litr"] - df["value"] / 1000).abs()

print("Nieprawidłowe przeliczenia:", (roznica > 0.000001).sum())
print("Największa różnica:", roznica.max())


Nieprawidłowe przeliczenia: 0
Największa różnica: 0.0
